# 41. Intervals for the Table 7 complement sets and fragility of the 24-hour ranking reversal

Here I address two questions about the 24-hour ranking reversal on Binance spot BTCUSDT one-minute data, over the development period only (2023-01 to 2026-04). This analysis is exploratory; the preregistered plans (OSF TB245, BKJUQ) stay unchanged.

1. What are the intervals for the complement sets of Table 7? In Table 7 I derived the GBM − M0 standard AUC difference of +0.083 on the 166 events found only under the 24-hour definition, and of +0.024 on the 31 events found only under the 4-hour definition, algebraically from stored values ("interval not computed"). Here I compute both directly on the same minutes and scores, with 7-day block bootstrap and event bootstrap intervals (sections 1–2; `1_guards.csv`, `2_subset_intervals.csv`).
2. How many events does the 24-hour ranking reversal rest on? Over 218 events GBM leads on standard AUC (+0.057) and M0 leads on pre-onset AUC (−0.090), but only 30 events have pre-onset minutes. I remove the most influential events one at a time (greedy, at most 40 steps) and record the number k at which the 95% intervals include zero, for the standard, pre-onset and within-regime pre-onset AUC differences of the reversal and for the standard AUC difference of the 166 complement events (section 3; `3_fragility_path.csv`, `3b_fragility_summary.csv`, `3c_leave_one_out.csv`).

All differences are GBM − M0. A subset (all, common, other) restricts only the positives to its events and leaves the negatives unchanged (the Table 7 convention). For the intervals I use 2,000 block draws and 2,000 event draws. I call a difference distinguishable only when the 7-day block interval and the event bootstrap interval (a: events resampled with replacement, negatives fixed) both exclude zero on the same side; I report variant (b), which also resamples negatives by 7-day blocks, for reference only. On a fragility path, k_joint is the first step at which both intervals include zero at the same step; since I find it in greedy order, it is an upper bound on the smallest number of events whose removal achieves this. I choose the removed events after seeing the results, so section 3 is an exploratory sensitivity analysis, and I do not read its intervals as inference. I fix the reading rules before the results and record them in `41_SPEC.json`.

In section 0 I rebuild the scores, labels, blocks and deciles with the code of section 0 of notebooks 32, 33 and 36. The inputs are the spot cache (`cache/`), the notebook 02 event catalog and the notebook 16 frozen models (`methods_definitions_v1`). I write all results, including `41_SUMMARY.json` and `41_SPEC.json`, only to `complement_fragility_v1`; they contain no prices (times, counts, AUCs and intervals only).

In [1]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

# Shared with notebook 16 (identical code): VPIN, causal feature set, GBM settings.
REGIME_WINDOW_D = 30; RV_WINDOW = 1440; VPIN_BUCKETS_PER_DAY = 50; VPIN_WINDOW = 50; VPIN_LOOKBACK_D = 30
GBM_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                  class_weight='balanced', early_stopping=False, random_state=17)
FEATS = ['z_vol', 'tsr', 'z_hs', 'ret_5', 'ret_15', 'ret_60', 'ret_240', 'rv_15', 'rv_60', 'rv_240', 'rv_ratio_15_240',
         'dd_240', 'dd_1440', 'vol_intensity_15', 'sell_ratio_15', 'sell_ratio_60', 'vpin']

@njit(cache=False)
def _vpin_core(vol, buy, bucket, n_win):
    out = np.full(len(vol), np.nan); ring = np.zeros(n_win); k = 0; run = 0.0; cv = 0.0; cb = 0.0
    for i in range(len(vol)):
        v = vol[i]; b = buy[i]
        if not (v == v) or not (b == b) or v < 0:
            v = 0.0; b = 0.0
        cv += v; cb += min(max(b, 0.0), v)
        bs = bucket[i]
        if bs == bs and bs > 0 and cv >= bs:
            imb = abs(2.0 * cb - cv) / cv
            j = k % n_win; run += imb - ring[j]; ring[j] = imb; k += 1; cv = 0.0; cb = 0.0
        if k >= n_win:
            out[i] = run / n_win
    return out


def vpin_series(px):
    vol = px.volume.to_numpy(float); buy = px.taker_buy_vol.to_numpy(float)
    daily = px.volume.groupby(px.index.floor('D')).sum()
    size = (daily.rolling(VPIN_LOOKBACK_D, min_periods=VPIN_LOOKBACK_D).mean().shift(1) / VPIN_BUCKETS_PER_DAY)
    bucket = size.reindex(px.index.floor('D')).to_numpy(float)
    return _vpin_core(vol, buy, bucket, VPIN_WINDOW)


def causal_features(px, sg):
    c = px.close.astype(float); lc = np.log(c); r = lc.diff(); vol = px.volume.astype(float); sell = vol - px.taker_buy_vol.astype(float)
    F = pd.DataFrame(index=px.index)
    F['z_vol'] = sg.z_vol.astype(float); F['tsr'] = sg.tsr.astype(float); F['z_hs'] = sg.z_hs.astype(float)
    for k in (5, 15, 60, 240):
        F[f'ret_{k}'] = lc - lc.shift(k)
    for k in (15, 60, 240):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k // 2).std()
    F['rv_ratio_15_240'] = F.rv_15 / F.rv_240
    F['dd_240'] = 1 - c / c.rolling(240, min_periods=120).max()
    F['dd_1440'] = 1 - c / c.rolling(1440, min_periods=720).max()
    F['vol_intensity_15'] = vol.rolling(15, min_periods=8).sum() / (vol.rolling(1440, min_periods=720).sum() / 96)
    F['sell_ratio_15'] = sell.rolling(15, min_periods=8).sum() / vol.rolling(15, min_periods=8).sum()
    F['sell_ratio_60'] = sell.rolling(60, min_periods=30).sum() / vol.rolling(60, min_periods=30).sum()
    F['vpin'] = vpin_series(px)
    return F.replace([np.inf, -np.inf], np.nan).astype('float32')
# Shared by notebooks 18 and 19: exact block-bootstrap AUC decomposition.
def week_pairs(sp, wp, sn, wn):
    """C[k, l] = over positives in block k, the number of negatives in block l with a lower score (+0.5 per tie).
    With block multiplicities w from the bootstrap, the AUC is w'Cw / ((w'P)(w'N)); w = 1 gives the ordinary AUC."""
    C = np.zeros((N_WEEKS, N_WEEKS))
    if len(sp) == 0 or len(sn) == 0:
        return C
    o = np.lexsort((sn, wn)); sn = sn[o]; wn = wn[o]
    bounds = np.searchsorted(wn, np.arange(N_WEEKS + 1), 'left')
    for l in range(N_WEEKS):
        a, b = bounds[l], bounds[l + 1]
        if a == b:
            continue
        seg = sn[a:b]; lo = np.searchsorted(seg, sp, 'left'); hi = np.searchsorted(seg, sp, 'right')
        C[:, l] = np.bincount(wp, weights=lo + 0.5 * (hi - lo), minlength=N_WEEKS)
    return C


def _ratio(a, b):
    return a / b if b > 0 else np.nan


STAT_KEYS = ['auc_standard', 'auc_pre', 'auc_in', 'share_in', 'diff_standard_minus_pre', 'in_share_of_excess', 'auc_calm',
             'auc_standard_within_regime', 'auc_pre_within_regime', 'auc_in_within_regime']


def make_parts(sm, code, calm, wk, dec):
    neg = code == 0; pre = code == 1; inn = code == 2; cnt = lambda k: np.bincount(wk[k], minlength=N_WEEKS).astype(float)
    P = dict(C_pre=week_pairs(sm[pre], wk[pre], sm[neg], wk[neg]), C_in=week_pairs(sm[inn], wk[inn], sm[neg], wk[neg]),
             C_calm=week_pairs(sm[(code > 0) & calm], wk[(code > 0) & calm], sm[neg & calm], wk[neg & calm]),
             P_pre=cnt(pre), P_in=cnt(inn), N=cnt(neg), P_calm=cnt((code > 0) & calm), N_calm=cnt(neg & calm),
             Cs_pre=np.zeros((N_WEEKS, N_WEEKS)), Cs_in=np.zeros((N_WEEKS, N_WEEKS)), Pq_pre=[], Pq_in=[], Nq=[])
    for q in range(10):
        kq = dec == q; nq = neg & kq; pq = pre & kq; iq = inn & kq
        P['Cs_pre'] += week_pairs(sm[pq], wk[pq], sm[nq], wk[nq]); P['Cs_in'] += week_pairs(sm[iq], wk[iq], sm[nq], wk[nq])
        P['Pq_pre'].append(cnt(pq)); P['Pq_in'].append(cnt(iq)); P['Nq'].append(cnt(nq))
    for k in ('Pq_pre', 'Pq_in', 'Nq'):
        P[k] = np.array(P[k])
    return P


def stats_from(P, w):
    q = lambda C: w @ C @ w
    Wn = w @ P['N']; Wp = w @ P['P_pre']; Wi = w @ P['P_in']; sp = q(P['C_pre']); si = q(P['C_in'])
    std = _ratio(sp + si, (Wp + Wi) * Wn); pre = _ratio(sp, Wp * Wn); inn = _ratio(si, Wi * Wn); pi = _ratio(Wi, Wp + Wi)
    nq = P['Nq'] @ w; pq = P['Pq_pre'] @ w; iq = P['Pq_in'] @ w; ssp = q(P['Cs_pre']); ssi = q(P['Cs_in'])
    return dict(auc_standard=std, auc_pre=pre, auc_in=inn, share_in=pi, diff_standard_minus_pre=std - pre,
                in_share_of_excess=_ratio(pi * (inn - 0.5), std - 0.5) if np.isfinite(inn) else 0.0,
                auc_calm=_ratio(q(P['C_calm']), (w @ P['P_calm']) * (w @ P['N_calm'])),
                auc_standard_within_regime=_ratio(ssp + ssi, ((pq + iq) * nq).sum()),
                auc_pre_within_regime=_ratio(ssp, (pq * nq).sum()), auc_in_within_regime=_ratio(ssi, (iq * nq).sum()))
import joblib

OUT = SPOT_BASE / 'complement_fragility_v1'; OUT.mkdir(parents=True, exist_ok=True)
M16 = SPOT_BASE / 'methods_definitions_v1'
M22 = SPOT_BASE / 'peak_window_v1'
M32 = SPOT_BASE / 'robustness_checks_v1'
M33 = SPOT_BASE / 'shapley_decomposition_v1'
M36 = SPOT_BASE / 'review_r7_checks_v1'
DNAME, DCL = '4pct', 0.04
H_REF, H_24 = 240, 1440; H_LIST = [H_REF, H_24]
PSCORES = ['M0_VF_logit', 'M1_vol_threshold', 'M3_GBM_rich', 'B0_RV24h_baseline']        # common-minute set of notebooks 29, 32, 33, 36 (Tables 6-7)
A_NAME, B_NAME = 'M3_GBM_rich', 'M0_VF_logit'                                              # every difference is A - B (GBM - M0)
SEED = 20261141
SEED32 = 20261031; N_BOOT32 = 500                    # notebook-32 section-3 draws (seed + 3), reproduced as a guard
SEED36 = 20261104; N_EVBOOT36 = 2000                 # notebook-36 section-3 draws (seeds + 3 + H, + 4 + H), reproduced as a guard
N_BLOCK = 2000; N_EVENT = 2000; BLOCK_DAYS = 7
COMMON_TOL_MIN = 60                                  # notebook 32
MIN_PRE_EVENTS = 5                                   # section 2: fewer events with pre-onset minutes -> "not interpretable"
K_MAX = 40                                           # section 3: greedy steps at most
MIN_FINITE = 0.95                                    # an interval is judged only if both ends are finite and at least this share of draws is finite
QUANTS = ['auc_standard', 'auc_pre', 'auc_in', 'auc_pre_within_regime']
PRE_QUANTS = ('auc_pre', 'auc_pre_within_regime')
FRAG_TARGETS = [(H_24, 'all', 'auc_standard'), (H_24, 'all', 'auc_pre'), (H_24, 'all', 'auc_pre_within_regime'), (H_24, 'other', 'auc_standard')]
MOCK_BYPASS = ()                                     # empty on Colab; only the synthetic test run lists guard names here, and every bypass is printed and recorded
EVAL_LAST = ts(EVAL_END).value - (LEAD_MAX + 5) * NS
GUARDS = []


def guard(name, ok, msg):
    """Record a guard; raise RuntimeError('GUARD FAIL: ...') unless it holds. A guard listed in MOCK_BYPASS (synthetic test run only) is reported, not raised."""
    ok = bool(ok); byp = (not ok) and (name in MOCK_BYPASS)
    GUARDS.append(dict(guard=name, passed=ok, bypassed_in_mock=byp, detail=str(msg)))
    if ok:
        return True
    if byp:
        print(f'Guard "{name}" bypassed: {msg} (synthetic test run only)', flush=True)
        return False
    raise RuntimeError(f'GUARD FAIL: {msg}')


for _p in [M16 / 'METHODS_FROZEN.json', M16 / 'METHODS_SPEC.json', M22 / '2_per_minute_by_window.csv', M32 / '3_common_events.csv', M32 / '3b_common_event_pairs.csv',
           M33 / '1_decomposition_point.csv', M36 / '3_event_bootstrap.csv']:
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 16, 22, 32, 33 and 36 are required: {_p}')
FROZEN16 = json.loads((M16 / 'METHODS_FROZEN.json').read_text()); SPEC16 = json.loads((M16 / 'METHODS_SPEC.json').read_text())
if sha256_file(M16 / 'METHODS_SPEC.json') != FROZEN16['spec_sha256'] or SPEC16['gbm_params'] != GBM_PARAMS or SPEC16['gbm_features'] != FEATS:
    raise RuntimeError('GUARD FAIL: notebook-16 spec/settings differ')

SPEC = {'title': 'Table 7 complement intervals and fragility of the 24-hour ranking reversal (exploratory; development sample)',
        'inputs': 'spot cache, notebook-2 catalog, notebook-16 frozen models; stored results of notebooks 22, 32, 33, 36 for guards; holdouts and the prospective folder not read',
        'minutes': 'notebook-33 common minutes (valid, finite M0, M1, GBM, B0 scores, RV decile assigned, evaluation period)',
        'subsets': {'all': 'every development event of the definition', 'common': f'events paired across the 4-hour and 24-hour definitions (breaches within {COMMON_TOL_MIN} min, notebook-32 pairing)',
                    'other': 'events of the definition that have no counterpart under the other definition (exact complement of common)',
                    'labels': 'positives kept only for the subset events; negatives unchanged (Table 7 convention)'},
        'quantities': {q: 'GBM - M0' for q in QUANTS},
        'intervals': {'block': f'{N_BLOCK} draws, {BLOCK_DAYS}-day blocks, shared by both systems (paired)',
                      'event_a': f'{N_EVENT} draws: subset events resampled with replacement (all positive minutes of a drawn event, times its multiplicity), negatives fixed',
                      'event_b': 'event_a draws plus negatives resampled by 7-day blocks (independent draws)', 'level': '95% percentile intervals over finite draws'},
        'reading_section2': f'distinguishable (+/-) if the block interval and the event_a interval both exclude zero on the same side; otherwise not distinguishable; '
                            f'pre-onset and within-regime pre-onset differences are "not interpretable" when fewer than {MIN_PRE_EVENTS} subset events have pre-onset minutes; '
                            f'"not interpretable" also when either interval has a non-finite end or fewer than {MIN_FINITE:.0%} finite draws',
        'fragility': {'targets': [f'H={h} {s} {q}' for h, s, q in FRAG_TARGETS],
                      'greedy': 'at each step remove the remaining event whose removal gives the smallest sign(initial point) x point estimate (negatives fixed); recompute the point estimate, '
                                'the block interval (the section-2 block draws at every step) and the event_a interval (step 0: the section-2 event draws; later steps: new draws among the remaining events, seed fixed per step)',
                      'k': 'k_joint = first step at which the block and the event_a 95% intervals both include zero at that same step (the reading uses k_joint only); '
                           'k_block, k_event = first step at which each interval includes zero on its own, and k_sign = first step at which the point estimate changes sign, reported only; step 0 = nothing removed',
                      'interval_status': f'an interval is undetermined (never counted as including zero) if an end is not finite or fewer than {MIN_FINITE:.0%} of draws are finite; the path stops at an undetermined interval',
                      'stop_pre': f'for pre-onset and within-regime pre-onset targets the path stops as not interpretable once fewer than {MIN_PRE_EVENTS} remaining events have pre-onset minutes',
                      'bound': 'greedy k_joint is an upper bound on the smallest number of events whose removal makes both intervals include zero', 'label': 'k_joint <= 2 is described as resting on one or two events',
                      'caveat': 'events are removed after looking at their influence, so section 3 is an exploratory sensitivity analysis; intervals after removal are not inference',
                      'max_steps': K_MAX},
        'guards': 'notebook-22 per-minute AUCs; notebook-33 GBM - M0 points; notebook-32 pairs, Table-7 points and 500-draw intervals; notebook-36 event-bootstrap intervals; '
                  'complement points equal the algebraic complement of the notebook-32 rows',
        'seed': SEED, 'notebook16_spec_sha256': FROZEN16['spec_sha256'], 'holdouts': 'not used', 'prospective_folder': 'not read',
        'status': 'exploratory; development sample only; no confirmatory plan changed'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode()
_sp = OUT / '41_SPEC.json'
if _sp.exists() and _sp.read_bytes() != _raw:
    _sp.rename(OUT / f'41_SPEC.replaced_{pd.Timestamp.now(tz="UTC"):%Y%m%dT%H%M%S}.json')
_sp.write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
print('Output:', OUT, '| spec sha256', SPEC_SHA[:16], '| skipped checks:', MOCK_BYPASS if MOCK_BYPASS else 'none')

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/complement_fragility_v1 | spec sha256 a61fa8887a1152ff | skipped checks: none


## 0. Reproduce scores, labels, blocks and deciles

I use the same code as section 0 of notebooks 32, 33 and 36.

In [2]:
t_ = time.time()
px, sg = load_btc_spot()
F = causal_features(px, sg)
T = px.index.asi8; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v_ = sg.z_vol.to_numpy(float); f_ = sg.tsr.to_numpy(float); h_ = sg.z_hs.to_numpy(float)
VALID = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v_) & np.isfinite(f_) & np.isfinite(h_) & (f_ >= 0) & (f_ <= 1))
EVM = VALID & (T >= ts(EVAL_START).value) & (T < EVAL_LAST)
DD240 = F.dd_240.to_numpy(float); CALM = np.isfinite(DD240) & (DD240 < DCL / 4)
_lc = np.log(np.where(np.isfinite(close) & (close > 0), close, np.nan))
RV = pd.Series(_lc).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
DEC = np.full(len(T), -1, np.int64); _okr = EVM & np.isfinite(RV)
DEC[_okr] = np.minimum((pd.Series(RV[_okr]).rank(pct=True).to_numpy() * 10).astype(int), 9)
WEEK = np.where(EVM, (T - ts(EVAL_START).value) // (BLOCK_DAYS * 1440 * NS), -1).astype(np.int64)
N_WEEKS = int(WEEK[EVM].max()) + 1
TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']


def catalog_H(H):
    E_ = price_only_catalog(close, volume, px.index, H=H, decline=DCL)
    for c_ in TCOLS:
        E_[c_] = pd.to_datetime(E_[c_], utc=True)
    return E_.sort_values('breach_utc').reset_index(drop=True)


def in_decline_mask(E_, y_):
    br = E_.breach_utc.astype('int64').to_numpy(); on = E_.onset_1pct_utc.astype('int64').to_numpy()
    pos = np.searchsorted(br, T + LEAD_MIN * NS, 'left'); ok = pos < len(br)
    inn = np.zeros(len(T), bool); inn[ok] = T[ok] >= on[pos[ok]]
    return inn & (y_ == 1)


def event_of_minute(E_, y_):
    """Index (row) of the event that makes each positive minute positive; -1 elsewhere."""
    br = E_.breach_utc.astype('int64').to_numpy(); pos = np.searchsorted(br, T + LEAD_MIN * NS, 'left')
    out = np.full(len(T), -1, np.int64); ok = (y_ == 1) & (pos < len(br)); out[ok] = pos[ok]
    return out


E_ref = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
CAT = {H: catalog_H(H) for H in H_LIST}
guard('catalog_nb2', len(CAT[H_REF]) == len(E_ref) and all(np.array_equal(CAT[H_REF][c_].astype('int64').to_numpy(), E_ref[c_].astype('int64').to_numpy()) for c_ in TCOLS),
      'H=240 catalog differs from the notebook-2 catalog')
EV = {H: events_between(CAT[H], EVAL_START, EVAL_END).reset_index(drop=True) for H in H_LIST}
Xl = F[['z_vol', 'tsr']].to_numpy(float); okl = train_mask(T, VALID) & np.isfinite(Xl).all(axis=1)
cfg0 = FROZEN16['configs'][f'M0_VF_logit|{DNAME}']
m0 = new_logit().fit(Xl[okl], label_from_catalog(T, CAT[H_REF])[okl])
guard('m0_refit_nb16', np.allclose(m0[-1].coef_.ravel(), cfg0['coef'], rtol=1e-6, atol=1e-9) and np.isclose(m0[-1].intercept_[0], cfg0['intercept'], rtol=1e-6, atol=1e-9),
      'M0 refit differs from notebook 16')
cfg3 = FROZEN16['configs'][f'M3_GBM_rich|{DNAME}']; mp = SPOT_BASE / cfg3['model_file']
guard('gbm_sha_nb16', sha256_file(mp) == cfg3['model_sha256'], 'notebook-16 GBM file SHA-256 differs')
g3 = joblib.load(mp)
IDX = np.flatnonzero(EVM); S = {}
s = np.full(len(T), np.nan); okp = EVM & np.isfinite(Xl).all(axis=1); s[okp] = m0.predict_proba(Xl[okp])[:, 1]; S['M0_VF_logit'] = s
S['M1_vol_threshold'] = np.where(VALID, F.z_vol.to_numpy(float), np.nan)
s = np.full(len(T), np.nan); s[IDX] = g3.predict_proba(F[FEATS].to_numpy(np.float32)[IDX])[:, 1]; S['M3_GBM_rich'] = s
S['B0_RV24h_baseline'] = np.where(VALID, RV, np.nan)
del Xl, F, sg, g3

PM22 = pd.read_csv(M22 / '2_per_minute_by_window.csv'); ONES = np.ones(N_WEEKS); LAB = {}; EVOF = {}
for H in H_LIST:
    yH = label_from_catalog(T, CAT[H]); innH = in_decline_mask(CAT[H], yH)
    LAB[H] = np.where(yH == 1, np.where(innH, 2, 1), 0).astype(np.int64); EVOF[H] = event_of_minute(CAT[H], yH)
    for sname in PSCORES:
        m = EVM & np.isfinite(S[sname]) & (DEC >= 0)
        pt = stats_from(make_parts(S[sname][m], LAB[H][m], CALM[m], WEEK[m], DEC[m]), ONES)
        ref = PM22[(PM22.H == H) & (PM22.score == sname)]
        dmax = max(abs(pt[k] - float(ref.iloc[0][k])) for k in ['auc_standard', 'auc_pre', 'auc_in', 'auc_pre_within_regime']) if len(ref) == 1 else np.inf
        guard('per_minute_nb22', dmax <= 1e-9, f'per-minute point estimate differs from notebook 22 (H={H}, {sname}) by {dmax:.2e}')
MC = EVM & (DEC >= 0)
for sname in PSCORES:
    MC &= np.isfinite(S[sname])
print(f'PASS: notebook-16 models verified; per-minute point estimates equal notebook 22 at 4 h and 24 h ({time.time() - t_:.0f}s)')
print('development events:', {H: len(EV[H]) for H in H_LIST}, '| common valid minutes:', int(MC.sum()))

PP_MC = {H: {sn: make_parts(S[sn][MC], LAB[H][MC], CALM[MC], WEEK[MC], DEC[MC]) for sn in [A_NAME, B_NAME]} for H in H_LIST}
PT33 = pd.read_csv(M33 / '1_decomposition_point.csv'); PT33 = PT33[(PT33.h1 == H_REF) & (PT33.h2 == H_24)].iloc[0]
Q0 = {}
for H, tag in ((H_REF, 'h1'), (H_24, 'h2')):
    ra = stats_from(PP_MC[H][A_NAME], ONES); rb = stats_from(PP_MC[H][B_NAME], ONES)
    Q0[H] = dict(pi=ra['share_in'], D=ra['auc_standard'] - rb['auc_standard'], dpre=ra['auc_pre'] - rb['auc_pre'], din=ra['auc_in'] - rb['auc_in'])
dmax = max(abs(Q0[H][k] - float(PT33[f'{k}_{tag}'])) for H, tag in ((H_REF, 'h1'), (H_24, 'h2')) for k in ['pi', 'dpre', 'din', 'D'])
guard('nb33_point', dmax <= 1e-9, f'GBM - M0 point differences differ from notebook 33 by {dmax:.2e}')
print(f'PASS: GBM - M0 at 4 h: D {Q0[H_REF]["D"]:+.4f}, dpre {Q0[H_REF]["dpre"]:+.4f}; at 24 h: D {Q0[H_24]["D"]:+.4f}, dpre {Q0[H_24]["dpre"]:+.4f} (equal notebook 33)')
print(f'section 0 done ({time.time() - t_:.0f}s)')

PASS: notebook-16 models verified; per-minute point estimates equal notebook 22 at 4 h and 24 h (115s)
development events: {240: 83, 1440: 218} | common valid minutes: 1750612
PASS: GBM - M0 at 4 h: D -0.0081, dpre -0.0189; at 24 h: D +0.0568, dpre -0.0903 (equal notebook 33)
section 0 done (135s)


## 1. Common-event pairs, (event × 7-day block) calculator

I group positive minutes into units of (the event that makes them positive, 7-day block) and negatives into blocks. With unit weights a and negative-block weights w, the block bootstrap (a = multiplicity of the unit's block) and the event bootstrap (a = multiplicity of the unit's event) then use the same matrices, and a subset multiplies a by 0/1.

In [3]:
t_ = time.time()
ev4, ev24 = EV[H_REF], EV[H_24]
b4 = ev4.breach_utc.astype('int64').to_numpy(); b24 = ev24.breach_utc.astype('int64').to_numpy()
pairs = []; used24 = set()
for i4, b_ in enumerate(b4):                                        # pairing as in notebook 32
    j = int(np.argmin(np.abs(b24 - b_)))
    if abs(b24[j] - b_) <= COMMON_TOL_MIN * NS and j not in used24:
        used24.add(j); pairs.append((i4, j))
PR = pd.DataFrame([dict(event_id_H240=int(ev4.event_id[i]), event_id_H1440=int(ev24.event_id[j])) for i, j in pairs])
PR32 = pd.read_csv(M32 / '3b_common_event_pairs.csv')
guard('pairs_nb32', len(PR) == len(PR32) and np.array_equal(PR.event_id_H240.to_numpy(), PR32.event_id_H240.to_numpy()) and np.array_equal(PR.event_id_H1440.to_numpy(), PR32.event_id_H1440.to_numpy()),
      f'common-event pairs ({len(PR)}) differ from notebook 32 ({len(PR32)})')
ROW_OF = {H: {int(e): int(np.flatnonzero(CAT[H].event_id.to_numpy() == e)[0]) for e in CAT[H].event_id} for H in H_LIST}
COMMON_ROWS = {H_REF: np.array([ROW_OF[H_REF][int(e)] for e in PR.event_id_H240]), H_24: np.array([ROW_OF[H_24][int(e)] for e in PR.event_id_H1440])}
print(f'common events: {len(PR)} (equal notebook 32)')


def pair_counts(sp, gp, n_gp, sn, gn, n_gn):
    """notebook 36: C[k, l] = over positives in group k, the number of negatives in group l with a lower score (+0.5 per tie)."""
    C = np.zeros((n_gp, n_gn))
    if len(sp) == 0 or len(sn) == 0:
        return C
    o = np.lexsort((sn, gn)); sn = sn[o]; gn = gn[o]
    bounds = np.searchsorted(gn, np.arange(n_gn + 1), 'left')
    for l in range(n_gn):
        a, b = bounds[l], bounds[l + 1]
        if a == b:
            continue
        seg = sn[a:b]; lo = np.searchsorted(seg, sp, 'left'); hi = np.searchsorted(seg, sp, 'right')
        C[:, l] = np.bincount(gp, weights=lo + 0.5 * (hi - lo), minlength=n_gp)
    return C


def unit_parts(H):
    """Positives grouped by unit = (event, block) on the common minutes; negatives by block; within-regime counts by RV decile."""
    m = MC; code = LAB[H][m]; evc = EVOF[H][m]; wk = WEEK[m]; dec = DEC[m]
    pos = code > 0; neg = code == 0; pre = code == 1; inn = code == 2
    uni = np.unique(evc[pos]); n_ev = len(uni)
    key = np.full(len(code), -1, np.int64); key[pos] = np.searchsorted(uni, evc[pos]) * N_WEEKS + wk[pos]
    ukeys, uinv = np.unique(key[pos], return_inverse=True); n_u = len(ukeys)
    unit = np.full(len(code), -1, np.int64); unit[pos] = uinv
    cu = lambda k: np.bincount(unit[k], minlength=n_u).astype(float)
    U = dict(H=H, uni=uni, n_ev=n_ev, n_u=n_u, unit_ev=ukeys // N_WEEKS, unit_wk=ukeys % N_WEEKS, P_pre=cu(pre), P_in=cu(inn),
             N=np.bincount(wk[neg], minlength=N_WEEKS).astype(float),
             Pq_pre=np.array([cu(pre & (dec == q)) for q in range(10)]), Nq=np.array([np.bincount(wk[neg & (dec == q)], minlength=N_WEEKS) for q in range(10)]).astype(float))
    for sn in (A_NAME, B_NAME):
        s = S[sn][m]
        Cs = np.zeros((n_u, N_WEEKS))
        for q in range(10):
            kq = dec == q
            Cs += pair_counts(s[pre & kq], unit[pre & kq], n_u, s[neg & kq], wk[neg & kq], N_WEEKS)
        U[sn] = dict(C_pre=pair_counts(s[pre], unit[pre], n_u, s[neg], wk[neg], N_WEEKS), C_in=pair_counts(s[inn], unit[inn], n_u, s[neg], wk[neg], N_WEEKS), Cs_pre=Cs)
    # event-level sums (negatives fixed): used by the point estimates of section 3
    ev_sum = lambda x: np.bincount(U['unit_ev'], weights=x, minlength=n_ev)
    U['e_P_pre'] = ev_sum(U['P_pre']); U['e_P_in'] = ev_sum(U['P_in']); U['e_Pq_pre'] = np.array([ev_sum(U['Pq_pre'][q]) for q in range(10)])
    for sn in (A_NAME, B_NAME):
        for k in ('C_pre', 'C_in', 'Cs_pre'):
            U[sn]['e_' + k] = ev_sum(U[sn][k].sum(axis=1))
    return U


def _div(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    return np.divide(a, b, out=np.full(np.broadcast(a, b).shape, np.nan), where=b > 0)


def unit_stats(U, sn, A, Wm):
    """A: (draws, units) positive weights; Wm: (draws, blocks) negative weights. Returns arrays over draws."""
    P = U[sn]; A = np.atleast_2d(A); Wm = np.atleast_2d(Wm)
    Wn = Wm @ U['N']; Mp = A @ U['P_pre']; Mi = A @ U['P_in']
    sp = ((A @ P['C_pre']) * Wm).sum(1); si = ((A @ P['C_in']) * Wm).sum(1)
    swr = ((A @ P['Cs_pre']) * Wm).sum(1); dwr = ((A @ U['Pq_pre'].T) * (Wm @ U['Nq'].T)).sum(1)
    std = _div(sp + si, (Mp + Mi) * Wn); pre = _div(sp, Mp * Wn); inn = _div(si, Mi * Wn); pi = _div(Mi, Mp + Mi)
    return dict(auc_standard=std, auc_pre=pre, auc_in=inn, share_in=pi, auc_pre_within_regime=_div(swr, dwr),
                in_share_of_excess=_div(pi * (inn - 0.5), std - 0.5))


def unit_diff(U, A, Wm):
    ra = unit_stats(U, A_NAME, A, Wm); rb = unit_stats(U, B_NAME, A, Wm)
    return {k: ra[k] - rb[k] for k in QUANTS}, ra, rb


def interval(d):
    """notebook-33 interval(): 95% percentile interval over finite draws and the share of finite draws."""
    d = np.asarray(d, float); f = np.isfinite(d)
    lo, hi = (np.percentile(d[f], [2.5, 97.5]) if f.sum() else (np.nan, np.nan))
    return float(lo), float(hi), float(f.mean())


def zero_status(lo, hi, fin):
    """'excludes' / 'includes' zero, or 'undetermined' when an end is not finite or too few draws are finite (never counted as including zero)."""
    if not (np.isfinite(lo) and np.isfinite(hi)) or fin < MIN_FINITE:
        return 'undetermined'
    return 'excludes' if (lo > 0 or hi < 0) else 'includes'


def ev_keep(U, subset):
    """0/1 per event (order of U['uni']) for subset all / common / other."""
    if subset == 'all':
        return np.ones(U['n_ev'])
    c = np.isin(U['uni'], COMMON_ROWS[U['H']]).astype(float)
    return c if subset == 'common' else 1 - c


def block_A(U, Wm, keep_e):
    return Wm[:, U['unit_wk']] * keep_e[U['unit_ev']][None, :]


def event_A(U, Mm, keep_e):
    return Mm[:, U['unit_ev']] * keep_e[U['unit_ev']][None, :]


UP = {H: unit_parts(H) for H in H_LIST}
print('units:', {H: (UP[H]['n_ev'], UP[H]['n_u']) for H in H_LIST}, f'({time.time() - t_:.0f}s)')

# ---- guard (i): all-ones weights equal make_parts / stats_from on the same minutes, for each subset
G1 = []
for H in H_LIST:
    for subset in ('all', 'common', 'other'):
        keep = ev_keep(UP[H], subset)
        code = LAB[H].copy()
        if subset != 'all':
            rows_keep = UP[H]['uni'][keep > 0]
            code[(code > 0) & ~np.isin(EVOF[H], rows_keep)] = -1
        m = MC & (code >= 0)
        for sn in (A_NAME, B_NAME):
            ref = stats_from(make_parts(S[sn][m], code[m], CALM[m], WEEK[m], DEC[m]), ONES)
            got = unit_stats(UP[H], sn, block_A(UP[H], ONES[None, :], keep), ONES[None, :])
            dmax = max(abs(float(got[k][0]) - ref[k]) for k in QUANTS + ['share_in'] if np.isfinite(ref[k]))
            G1.append(dict(H=H, subset=subset, score=sn, max_abs_diff=dmax))
dmax = max(r['max_abs_diff'] for r in G1)
guard('unit_ones_vs_make_parts', dmax <= 1e-9, f'unit calculator with all-ones weights differs from make_parts/stats_from by {dmax:.2e}')

# ---- guard (ii): notebook-32 Table 7 (points and 500-draw intervals with the notebook-32 draws)
CE32 = pd.read_csv(M32 / '3_common_events.csv')
_r32 = np.random.default_rng(SEED32 + 3)
W32 = np.array([np.bincount(_r32.integers(0, N_WEEKS, N_WEEKS), minlength=N_WEEKS).astype(float) for _ in range(N_BOOT32)])
G2 = []
for H in H_LIST:
    for subset, lab in (('all', 'all events'), ('common', 'common events only')):
        keep = ev_keep(UP[H], subset); ref = CE32[(CE32.H == H) & (CE32.subset == lab)].iloc[0]
        d1, ra1, rb1 = unit_diff(UP[H], block_A(UP[H], ONES[None, :], keep), ONES[None, :])
        dB, _, _ = unit_diff(UP[H], block_A(UP[H], W32, keep), W32)
        vals = dict(positives=float(keep[UP[H]['unit_ev']] @ (UP[H]['P_pre'] + UP[H]['P_in'])), share_in=float(rb1['share_in'][0]),
                    m0_standard=float(rb1['auc_standard'][0]), gbm_standard=float(ra1['auc_standard'][0]), m0_pre=float(rb1['auc_pre'][0]), gbm_pre=float(ra1['auc_pre'][0]))
        for k in ('auc_standard', 'auc_pre', 'auc_pre_within_regime'):
            lo, hi = np.nanpercentile(dB[k], [2.5, 97.5])
            vals[f'diff_{k}'] = float(d1[k][0]); vals[f'diff_{k}_lo'] = float(lo); vals[f'diff_{k}_hi'] = float(hi)
        dmax = max(abs(vals[c] - float(ref[c])) for c in vals)
        G2.append(dict(H=H, subset=subset, max_abs_diff=dmax))
        guard('table7_nb32', dmax <= 1e-9, f'Table 7 row H={H} {lab} differs from notebook 32 by {dmax:.2e}')

# ---- guard (iii): notebook-36 event bootstrap (all events; variants a and b; the notebook-36 draws)
EB36 = pd.read_csv(M36 / '3_event_bootstrap.csv'); G3 = []
QN36 = {'standard AUC': 'auc_standard', 'pre-onset AUC': 'auc_pre', 'pre-onset AUC within regime': 'auc_pre_within_regime'}
for H in H_LIST:
    U = UP[H]; n_ev = U['n_ev']; keep = np.ones(n_ev)
    rng_e = np.random.default_rng(SEED36 + 3 + H); rng_w = np.random.default_rng(SEED36 + 4 + H)
    Mm = np.empty((N_EVBOOT36, n_ev)); Ww = np.empty((N_EVBOOT36, N_WEEKS))
    for i in range(N_EVBOOT36):                                     # notebook-36 loop order: one event draw, then one block draw
        Mm[i] = np.bincount(rng_e.integers(0, n_ev, n_ev), minlength=n_ev); Ww[i] = np.bincount(rng_w.integers(0, N_WEEKS, N_WEEKS), minlength=N_WEEKS)
    for v, Wm in (('a', np.ones((N_EVBOOT36, N_WEEKS))), ('b', Ww)):
        dE, _, _ = unit_diff(U, event_A(U, Mm, keep), Wm)
        for qname, k in QN36.items():
            ref = EB36[(EB36.H == H) & (EB36.variant.str.startswith(v + ':')) & (EB36.quantity == qname)].iloc[0]
            lo, hi, _ = interval(dE[k]); dmax = max(abs(lo - float(ref.lo)), abs(hi - float(ref.hi)))
            G3.append(dict(H=H, variant=v, quantity=k, max_abs_diff=dmax))
            guard('event_bootstrap_nb36', dmax <= 1e-9, f'event bootstrap H={H} variant {v} {qname} differs from notebook 36 by {dmax:.2e}')
del Mm, Ww

# ---- guard (iv): complement points equal the algebraic complement of the stored notebook-32 rows (formula of the stand-alone script behind Table 7)
G4 = []
for H in H_LIST:
    a = CE32[(CE32.H == H) & (CE32.subset == 'all events')].iloc[0]; c = CE32[(CE32.H == H) & (CE32.subset == 'common events only')].iloc[0]
    n_a, n_c = int(a.positives), int(c.positives); n_o = n_a - n_c
    pre_a = round(n_a * (1 - a.share_in)); pre_c = round(n_c * (1 - c.share_in)); pre_o = pre_a - pre_c
    comp = lambda col: (n_a * a[col] - n_c * c[col]) / n_o
    comp_pre = lambda col: (pre_a * a[col] - pre_c * c[col]) / pre_o if pre_o > 0 else np.nan
    keep = ev_keep(UP[H], 'other')
    _, ra, rb = unit_diff(UP[H], block_A(UP[H], ONES[None, :], keep), ONES[None, :])
    got = dict(m0_standard=rb['auc_standard'][0], gbm_standard=ra['auc_standard'][0], m0_pre=rb['auc_pre'][0], gbm_pre=ra['auc_pre'][0])
    alg = dict(m0_standard=comp('m0_standard'), gbm_standard=comp('gbm_standard'), m0_pre=comp_pre('m0_pre'), gbm_pre=comp_pre('gbm_pre'))
    dmax = max(abs(float(got[k]) - float(alg[k])) for k in got if np.isfinite(alg[k]))
    npos = float(keep[UP[H]['unit_ev']] @ (UP[H]['P_pre'] + UP[H]['P_in']))
    G4.append(dict(H=H, positives_other=npos, positives_other_algebraic=n_o, max_abs_diff=dmax))
    guard('complement_algebraic', dmax <= 1e-8 and npos == n_o, f'complement H={H}: points differ from the algebraic complement by {dmax:.2e} (positives {npos:.0f} vs {n_o})')
GD = pd.concat([pd.DataFrame(G1).assign(check='i_ones'), pd.DataFrame(G2).assign(check='ii_nb32_table7'), pd.DataFrame(G3).assign(check='iii_nb36_event_bootstrap'),
                pd.DataFrame(G4).assign(check='iv_complement_algebraic')], ignore_index=True)
GD.to_csv(OUT / '1_guards.csv', index=False)
print(f'PASS (i)-(iv): unit calculator equals make_parts; Table 7 points and 500-draw intervals equal notebook 32; event-bootstrap intervals equal notebook 36; '
      f'complements equal the algebraic values ({time.time() - t_:.0f}s)')

common events: 52 (equal notebook 32)
units: {240: (83, 83), 1440: (218, 218)} (11s)
PASS (i)-(iv): unit calculator equals make_parts; Table 7 points and 500-draw intervals equal notebook 32; event-bootstrap intervals equal notebook 36; complements equal the algebraic values (60s)


## 2. GBM − M0 differences and intervals by subset and definition (2,000 block draws; 2,000 event draws for a and b)

In [4]:
t_ = time.time()
rng_b = np.random.default_rng(SEED + 1)
WB = np.array([np.bincount(rng_b.integers(0, N_WEEKS, N_WEEKS), minlength=N_WEEKS).astype(float) for _ in range(N_BLOCK)])
rows = []; SUBSET_INDEX = {'all': 0, 'common': 1, 'other': 2}
for H in H_LIST:
    U = UP[H]; n_ev = U['n_ev']
    for subset, si_ in SUBSET_INDEX.items():
        keep = ev_keep(U, subset); sub = np.flatnonzero(keep > 0); n_sub = len(sub)
        n_pre_ev = int(((U['e_P_pre'] > 0) & (keep > 0)).sum())
        d1, ra1, rb1 = unit_diff(U, block_A(U, ONES[None, :], keep), ONES[None, :])
        dB, _, _ = unit_diff(U, block_A(U, WB, keep), WB)
        rng_e = np.random.default_rng(SEED + 100 + 10 * si_ + H); rng_w = np.random.default_rng(SEED + 200 + 10 * si_ + H)
        Mm = np.zeros((N_EVENT, n_ev)); Ww = np.empty((N_EVENT, N_WEEKS))
        for i in range(N_EVENT):
            Mm[i, sub] = np.bincount(rng_e.integers(0, n_sub, n_sub), minlength=n_sub); Ww[i] = np.bincount(rng_w.integers(0, N_WEEKS, N_WEEKS), minlength=N_WEEKS)
        dEa, _, _ = unit_diff(U, event_A(U, Mm, keep), np.ones((N_EVENT, N_WEEKS)))
        dEb, _, _ = unit_diff(U, event_A(U, Mm, keep), Ww)
        npre_draw = ((Mm > 0) & (U['e_P_pre'] > 0)[None, :]).sum(1)
        for k in QUANTS:
            p_ = float(d1[k][0]); bl, bh, bf = interval(dB[k]); al, ah, af = interval(dEa[k]); cl, ch, cf = interval(dEb[k])
            if k in PRE_QUANTS and n_pre_ev < MIN_PRE_EVENTS:
                reading = f'not interpretable ({n_pre_ev} events with pre-onset minutes)'
            elif zero_status(bl, bh, bf) == 'undetermined' or zero_status(al, ah, af) == 'undetermined':
                reading = f'not interpretable (finite draws: block {bf:.0%}, event {af:.0%})'
            elif (bl > 0 and al > 0) or (bh < 0 and ah < 0):
                reading = 'distinguishable (' + ('+' if bl > 0 else '-') + ')'
            else:
                reading = 'not distinguishable'
            rows.append(dict(H=H, subset=subset, events=n_sub, events_with_pre_onset_minutes=n_pre_ev,
                             positives=float(keep[U['unit_ev']] @ (U['P_pre'] + U['P_in'])), pre_onset_positives=float(keep[U['unit_ev']] @ U['P_pre']),
                             pi=float(rb1['share_in'][0]), quantity=k, m0=float(rb1[k][0]), gbm=float(ra1[k][0]), diff=p_,
                             block_lo=bl, block_hi=bh, block_finite=bf, event_a_lo=al, event_a_hi=ah, event_a_finite=af, event_b_lo=cl, event_b_hi=ch, event_b_finite=cf,
                             share_event_draws_without_pre_onset_event=float((npre_draw == 0).mean()) if k in PRE_QUANTS else np.nan, reading=reading))
        print(f'H={H:4d} {subset:6s} events {n_sub:3d} (with pre-onset minutes {n_pre_ev:2d}) | standard {d1["auc_standard"][0]:+.4f} '
              f'block [{rows[-4]["block_lo"]:+.4f}, {rows[-4]["block_hi"]:+.4f}] event [{rows[-4]["event_a_lo"]:+.4f}, {rows[-4]["event_a_hi"]:+.4f}] -> {rows[-4]["reading"]} ({time.time() - t_:.0f}s)', flush=True)
SI = pd.DataFrame(rows); SI.to_csv(OUT / '2_subset_intervals.csv', index=False)
display(SI[['H', 'subset', 'events', 'events_with_pre_onset_minutes', 'pi', 'quantity', 'diff', 'block_lo', 'block_hi', 'event_a_lo', 'event_a_hi', 'reading']].round(4))

H= 240 all    events  83 (with pre-onset minutes 83) | standard -0.0081 block [-0.0392, +0.0245] event [-0.0385, +0.0216] -> not distinguishable (0s)
H= 240 common events  52 (with pre-onset minutes 52) | standard -0.0270 block [-0.0668, +0.0154] event [-0.0662, +0.0111] -> not distinguishable (1s)
H= 240 other  events  31 (with pre-onset minutes 31) | standard +0.0236 block [-0.0159, +0.0665] event [-0.0173, +0.0654] -> not distinguishable (1s)
H=1440 all    events 218 (with pre-onset minutes 30) | standard +0.0568 block [+0.0323, +0.0814] event [+0.0345, +0.0790] -> distinguishable (+) (1s)
H=1440 common events  52 (with pre-onset minutes 27) | standard -0.0283 block [-0.0679, +0.0150] event [-0.0701, +0.0113] -> not distinguishable (1s)
H=1440 other  events 166 (with pre-onset minutes  3) | standard +0.0835 block [+0.0561, +0.1089] event [+0.0576, +0.1098] -> distinguishable (+) (2s)


,H,subset,events,events_with_pre_onset_minutes,pi,quantity,diff,block_lo,block_hi,event_a_lo,event_a_hi,reading
0,240,all,83,83,0.4137,auc_standard,-0.0081,-0.0392,0.0245,-0.0385,0.0216,not distinguishable
1,240,all,83,83,0.4137,auc_pre,-0.0189,-0.0612,0.0256,-0.0635,0.0244,not distinguishable
2,240,all,83,83,0.4137,auc_in,0.0071,-0.0172,0.0327,-0.0139,0.0281,not distinguishable
3,240,all,83,83,0.4137,auc_pre_within_regime,-0.0456,-0.0939,0.0049,-0.0977,0.0086,not distinguishable
4,240,common,52,52,0.3576,auc_standard,-0.0270,-0.0668,0.0154,-0.0662,0.0111,not distinguishable
5,240,common,52,52,0.3576,auc_pre,-0.0424,-0.0979,0.0165,-0.1007,0.0123,not distinguishable
6,240,common,52,52,0.3576,auc_in,0.0004,-0.0248,0.0272,-0.0259,0.0241,not distinguishable
7,240,common,52,52,0.3576,auc_pre_within_regime,-0.0714,-0.1349,-0.0014,-0.1375,-0.0098,distinguishable (-)
8,240,other,31,31,0.5077,auc_standard,0.0236,-0.0159,0.0665,-0.0173,0.0654,not distinguishable
9,240,other,31,31,0.5077,auc_pre,0.0326,-0.0214,0.0903,-0.0251,0.0918,not distinguishable


## 3. Fragility: removing the most influential events first (three quantities of the 24-hour reversal and the standard AUC difference of the 166 complement events)

In [5]:
t_ = time.time()


def point_diff(U, keep_e, k):
    """GBM - M0 point estimate with event weights keep_e (negatives fixed)."""
    out = []
    for sn in (A_NAME, B_NAME):
        P = U[sn]; Nt = U['N'].sum()
        if k == 'auc_standard':
            out.append(_div(keep_e @ (P['e_C_pre'] + P['e_C_in']), (keep_e @ (U['e_P_pre'] + U['e_P_in'])) * Nt))
        elif k == 'auc_pre':
            out.append(_div(keep_e @ P['e_C_pre'], (keep_e @ U['e_P_pre']) * Nt))
        else:
            out.append(_div(keep_e @ P['e_Cs_pre'], (U['e_Pq_pre'] @ keep_e) @ U['Nq'].sum(1)))
    return float(out[0] - out[1])


def leave_one_out(U, keep_e, k):
    """Point estimate after removing each event (vectorized); nan for events already removed or when undefined."""
    res = []
    for sn in (A_NAME, B_NAME):
        P = U[sn]; Nt = U['N'].sum()
        if k == 'auc_standard':
            c = P['e_C_pre'] + P['e_C_in']; p = U['e_P_pre'] + U['e_P_in']; res.append(_div(keep_e @ c - c * keep_e, (keep_e @ p - p * keep_e) * Nt))
        elif k == 'auc_pre':
            c = P['e_C_pre']; p = U['e_P_pre']; res.append(_div(keep_e @ c - c * keep_e, (keep_e @ p - p * keep_e) * Nt))
        else:
            nq = U['Nq'].sum(1); c = P['e_Cs_pre']; dq = U['e_Pq_pre'].T @ nq       # per event: sum over deciles of its pre-onset count x negatives
            res.append(_div(keep_e @ c - c * keep_e, (U['e_Pq_pre'] @ keep_e) @ nq - dq * keep_e))
    d = res[0] - res[1]; d[keep_e == 0] = np.nan
    return d


# unit check of the vectorized leave-one-out against direct recomputation on random events
_u = np.random.default_rng(SEED + 5); UCHK = []
for H in H_LIST:
    U = UP[H]
    for k in ('auc_standard', 'auc_pre', 'auc_pre_within_regime'):
        kv = np.ones(U['n_ev']); loo = leave_one_out(U, kv, k)
        for e in _u.choice(U['n_ev'], min(8, U['n_ev']), replace=False):
            kk = kv.copy(); kk[e] = 0; direct = point_diff(U, kk, k)
            UCHK.append(abs(loo[e] - direct) if np.isfinite(direct) else (0.0 if not np.isfinite(loo[e]) else np.inf))
        ref = unit_diff(U, block_A(U, ONES[None, :], kv), ONES[None, :])[0][k][0]
        UCHK.append(abs(point_diff(U, kv, k) - ref))
guard('loo_unit_check', max(UCHK) <= 1e-12, f'vectorized leave-one-out differs from direct recomputation by {max(UCHK):.2e}')


def event_info(H, row):
    e = CAT[H].iloc[int(row)]
    return dict(event_id=int(e.event_id), breach_utc=str(e.breach_utc), onset_to_breach_minutes=int(e.onset_to_breach_minutes),
                common=bool(int(row) in set(COMMON_ROWS[H].tolist())))


PATH = []; SUMM = []; LOO = []
for ti, (H, subset, k) in enumerate(FRAG_TARGETS):
    U = UP[H]; keep = ev_keep(U, subset).copy(); n0 = int(keep.sum())
    p0 = point_diff(U, keep, k); sgn = np.sign(p0)
    loo0 = leave_one_out(U, keep, k)
    for j in np.flatnonzero(keep > 0):
        LOO.append(dict(target=f'H{H}|{subset}|{k}', **event_info(H, U['uni'][j]), pre_onset_positives=float(U['e_P_pre'][j]),
                        in_progress_positives=float(U['e_P_in'][j]), point_without_event=float(loo0[j]), change=float(loo0[j] - p0)))
    removed = []; k_block = k_event = k_joint = k_sign = None; stop = None
    for step in range(K_MAX + 1):
        if step > 0:
            loo = leave_one_out(U, keep, k); cand = np.where(np.isfinite(loo), sgn * loo, np.inf)
            if not np.isfinite(cand).any():
                stop = f'no event left whose removal gives a defined estimate after {len(removed)} removals'; break
            j = int(np.argmin(cand)); keep[j] = 0; removed.append(j)
        n_pre_rem = int(((U['e_P_pre'] > 0) & (keep > 0)).sum())
        if k in PRE_QUANTS and n_pre_rem < MIN_PRE_EVENTS:
            stop = f'not interpretable: fewer than {MIN_PRE_EVENTS} events with pre-onset minutes remain after {len(removed)} removals'
            PATH.append(dict(target=f'H{H}|{subset}|{k}', step=step, **({'removed_' + a: b for a, b in event_info(H, U['uni'][removed[-1]]).items()} if step > 0 else {}),
                             events_with_pre_onset_minutes_remaining=n_pre_rem, point=point_diff(U, keep, k), note='path stopped (pre-onset events)'))
            break
        pt_ = point_diff(U, keep, k)
        dB, _, _ = unit_diff(U, block_A(U, WB, keep), WB); bl, bh, bf = interval(dB[k])
        sub = np.flatnonzero(keep > 0)
        rng_s = np.random.default_rng(SEED + 100 + 10 * SUBSET_INDEX[subset] + H if step == 0 else SEED + 7000 + 100 * ti + step)   # step 0: the section-2 event draws
        Mm = np.zeros((N_EVENT, U['n_ev']))
        for i in range(N_EVENT):
            Mm[i, sub] = np.bincount(rng_s.integers(0, len(sub), len(sub)), minlength=len(sub))
        dEa, _, _ = unit_diff(U, event_A(U, Mm, keep), np.ones((N_EVENT, N_WEEKS))); al, ah, af = interval(dEa[k])
        st_b = zero_status(bl, bh, bf); st_e = zero_status(al, ah, af)
        if k_block is None and st_b == 'includes':
            k_block = step
        if k_event is None and st_e == 'includes':
            k_event = step
        if k_joint is None and st_b == 'includes' and st_e == 'includes':
            k_joint = step
        if k_sign is None and np.isfinite(pt_) and np.sign(pt_) != sgn:
            k_sign = step
        PATH.append(dict(target=f'H{H}|{subset}|{k}', step=step, **({'removed_' + a: b for a, b in event_info(H, U['uni'][removed[-1]]).items()} if step > 0 else {}),
                         removed_pre_onset_positives=float(U['e_P_pre'][removed[-1]]) if step > 0 else np.nan, events_with_pre_onset_minutes_remaining=n_pre_rem,
                         point=pt_, block_lo=bl, block_hi=bh, block_finite=bf, event_lo=al, event_hi=ah, event_finite=af, block_status=st_b, event_status=st_e,
                         both_include_zero=bool(st_b == 'includes' and st_e == 'includes')))
        if 'undetermined' in (st_b, st_e):
            stop = f'interval undetermined at step {step} (finite draws: block {bf:.0%}, event {af:.0%})'; break
        if k_joint is not None and k_sign is not None:
            break
    if k_joint is not None:
        label = ('both intervals include zero with no event removed' if k_joint == 0 else 'rests on one or two events' if k_joint <= 2
                 else f'both intervals include zero after removing {k_joint} events (greedy upper bound)')
    elif stop is not None:
        label = stop
    else:
        label = f'the two intervals do not include zero at the same step within {len(removed)} removals'
    SUMM.append(dict(target=f'H{H}|{subset}|{k}', events=n0, events_with_pre_onset_minutes=int(((U['e_P_pre'] > 0) & (ev_keep(U, subset) > 0)).sum()),
                     point=p0, k_joint=k_joint, k_block=k_block, k_event=k_event, k_sign=k_sign, steps_run=len(removed), stop_reason=stop,
                     loo_min=float(np.nanmin(loo0)), loo_max=float(np.nanmax(loo0)), label=label))
    print(f'{SUMM[-1]["target"]}: point {p0:+.4f}; k_joint {k_joint} (k_block {k_block}, k_event {k_event}), k_sign {k_sign} of {n0} events; {label} '
          f'(leave-one-out range [{SUMM[-1]["loo_min"]:+.4f}, {SUMM[-1]["loo_max"]:+.4f}]) ({time.time() - t_:.0f}s)', flush=True)
FP = pd.DataFrame(PATH); FP.to_csv(OUT / '3_fragility_path.csv', index=False)
FS = pd.DataFrame(SUMM); FS.to_csv(OUT / '3b_fragility_summary.csv', index=False)
LO = pd.DataFrame(LOO); LO.to_csv(OUT / '3c_leave_one_out.csv', index=False)
display(FS.round(4))

H1440|all|auc_standard: point +0.0568; k_joint 25 (k_block 24, k_event 25), k_sign None of 218 events; both intervals include zero after removing 25 events (greedy upper bound) (leave-one-out range [+0.0542, +0.0593]) (6s)
H1440|all|auc_pre: point -0.0903; k_joint 2 (k_block 1, k_event 2), k_sign 7 of 218 events; rests on one or two events (leave-one-out range [-0.1033, -0.0740]) (7s)
H1440|all|auc_pre_within_regime: point -0.1394; k_joint 5 (k_block 5, k_event 4), k_sign 11 of 218 events; both intervals include zero after removing 5 events (greedy upper bound) (leave-one-out range [-0.1583, -0.1233]) (9s)
H1440|other|auc_standard: point +0.0835; k_joint 36 (k_block 34, k_event 36), k_sign None of 166 events; both intervals include zero after removing 36 events (greedy upper bound) (leave-one-out range [+0.0802, +0.0869]) (18s)


,target,events,events_with_pre_onset_minutes,point,k_joint,k_block,k_event,k_sign,steps_run,stop_reason,loo_min,loo_max,label
0,H1440|all|auc_standard,218,30,0.0568,25,24,25,NaN,40,None,0.0542,0.0593,both intervals include zero after removing 25 ...
1,H1440|all|auc_pre,218,30,-0.0903,2,1,2,7.0,7,None,-0.1033,-0.0740,rests on one or two events
2,H1440|all|auc_pre_within_regime,218,30,-0.1394,5,5,4,11.0,11,None,-0.1583,-0.1233,both intervals include zero after removing 5 e...
3,H1440|other|auc_standard,166,3,0.0835,36,34,36,NaN,40,None,0.0802,0.0869,both intervals include zero after removing 36 ...


## 4. Summary

In [6]:
def _j(o):
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return str(o)


summary = dict(spec_sha256=SPEC_SHA, run_time=str(pd.Timestamp.now(tz='UTC').floor('s')), status=SPEC['status'],
               mock_bypass_used=bool(any(g['bypassed_in_mock'] for g in GUARDS)), guards=GUARDS,
               events=dict((str(H), len(EV[H])) for H in H_LIST), common_events=len(PR), common_minutes=int(MC.sum()),
               subset_intervals={f'H{r.H}|{r.subset}|{r.quantity}': dict(diff=r.diff, block=[r.block_lo, r.block_hi], event_a=[r.event_a_lo, r.event_a_hi],
                                                                           event_b=[r.event_b_lo, r.event_b_hi], events=int(r.events), events_with_pre_onset_minutes=int(r.events_with_pre_onset_minutes),
                                                                           reading=r.reading) for r in SI.itertuples()},
               fragility=FS.to_dict('records'), outputs=sorted(p.name for p in OUT.iterdir() if p.is_file()))
(OUT / '41_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=_j))
print(f'guards: {sum(g["passed"] for g in GUARDS)} passed, {sum(not g["passed"] for g in GUARDS)} failed'
      + (f' (bypassed in the mock: {sorted({g["guard"] for g in GUARDS if g["bypassed_in_mock"]})})' if summary['mock_bypass_used'] else ''))
print('Done. Outputs in', OUT)

guards: 33 passed, 0 failed
Done. Outputs in /content/drive/MyDrive/btc_lead_cost_results_77months_spot/complement_fragility_v1


## Interpretation
- In `2_subset_intervals.csv`, the row `H = 1440, subset = other, quantity = auc_standard` gives the interval for the "+0.083 (interval not computed)" entry of Table 7, and `reading` holds the pre-specified call. Under the 24-hour definition only a few complement (other) events have pre-onset minutes, so their pre-onset difference is mostly "not interpretable". The `all` and `common` rows carry the Table 7 point estimates; for their intervals I use 2,000 draws rather than 500, so they can differ in the third decimal.
- In `3b_fragility_summary.csv`, `k_joint` is the value I use for the reading; `k_block` and `k_event` are the first steps at which each interval includes zero on its own, `k_sign` the step at which the point estimate changes sign, and `stop_reason` the reason a path stopped early (fewer than 5 events with pre-onset minutes, or an undetermined interval). `3_fragility_path.csv` lists the removed events with their times and onset-to-breach gaps, and `3c_leave_one_out.csv` the leave-one-out values.
- Only events with pre-onset minutes (30 in the data) affect the pre-onset AUC difference, so I read it together with that count.